In [ ]:
#pip install requests
#This notebook was written with edditing and formatind assitance to both the code and document, from Clauge Sonnet 5.

In [ ]:

#Load Libraries
import json
import requests #APIs
import re #regular expressions
import pandas as pd
import time


#Input your Open Respository url.
#If you do not want to submit your repository url each time, paste you url in quotes after the equal symbol.
#repository = "https://umassamh.dspace7-test.openrepository.com/"
repository = "https://scholarworks.umass.edu/"

In [ ]:
# Hardcode the field names for the ardvark_data JSON structure
field_names = [
    "id",
    "url_dataset",
    "dct_title_s",
    "dct_alternative_sm",
    "dct_description_sm",
    "dct_language_sm",
    "gbl_displayNote_sm",
    "dct_creator_sm",
    "dct_publisher_sm",
    "schema_provider_s",
    "gbl_resourceClass_sm",
    "gbl_resourceType_sm",
    "dct_subject_sm",
    "dcat_theme_sm",
    "dct_temporal_sm",
    "dct_issued_s",
    "gbl_indexYear_im",
    "dct_spatial_sm",
    "locn_geometry",
    "dcat_bbox",
    "dcat_centroid",
    "box_westlimit",
    "box_eastlimit",
    "box_northlimit",
    "box_southlimit",
    "point_north",
    "point_east",
    "pcdm_memberOf_sm",
    "dct_license_sm",
    "dct_rights_sm",
    "dct_accessRights_s",
    "dct_format_s",
    "gbl_fileSize_s",
    "dct_references_s",
    "dct_identifier_sm",
    "gbl_georeferenced_b",
    "gbl_mdVersion_s",
    "gbl_mdModified_dt",
    "dspace_UUID",
    "url_download_1_link",
    "url_download_1_type"
]

# Note: build_ardvark_record() adds url_download_1_link/url_download_1_type,
# url_download_2_link/url_download_2_type, etc. dynamically -- one numbered
# pair per bitstream found in each item's ORIGINAL bundle. A given record may
# end up with more numbered pairs than are listed here if it has more than
# one bitstream.

# Create an empty dictionary with the extracted field names and empty string values
ardvark_data = {field: '' for field in field_names}

# Convert the dictionary to a JSON string with pretty printing
ardvark_data_json = json.dumps(ardvark_data, indent=4)

# Display the empty JSON structure
#print(ardvark_data_json)

In [ ]:
# Read mapping rules from dc_ard_map.txt (only needs to be loaded once)
mapping_rules = {}
with open('dc_ard_map.txt', 'r') as f:
    for line in f:
        line = line.strip()
        if line and ':' in line:
            source_field, target_field = line.split(':', 1)
            mapping_rules[source_field.strip()] = target_field.strip()
            


In [ ]:
#Functions used in script

def get_original_bundle_downloads(item_json):
    """Follow the item's bundles -> ORIGINAL bundle -> bitstreams -> content URL chain.

    Returns a list of (content_url, file_extension, size_bytes) tuples, one per
    bitstream in the ORIGINAL bundle, in the order returned by the API.
    """
    downloads = []

    bundles_href = item_json.get('_links', {}).get('bundles', {}).get('href')
    if not bundles_href:
        return downloads

    bundles_resp = requests.get(bundles_href)
    bundles_resp.raise_for_status()
    bundles = bundles_resp.json().get('_embedded', {}).get('bundles', [])

    original_bundle = next((b for b in bundles if b.get('name') == 'ORIGINAL'), None)
    if original_bundle is None:
        return downloads

    bitstreams_href = original_bundle.get('_links', {}).get('bitstreams', {}).get('href')
    if not bitstreams_href:
        return downloads

    bitstreams_resp = requests.get(bitstreams_href)
    bitstreams_resp.raise_for_status()
    bitstreams = bitstreams_resp.json().get('_embedded', {}).get('bitstreams', [])

    for bitstream in bitstreams:
        content_href = bitstream.get('_links', {}).get('content', {}).get('href', '')
        filename = bitstream.get('name', '')
        extension = filename.rsplit('.', 1)[-1] if '.' in filename else ''
        size_bytes = bitstream.get('sizeBytes', 0) or 0
        downloads.append((content_href, extension, size_bytes))

    return downloads


def map_metadata_to_ardvark(uuid, metadata, downloads):
    """Map a raw dspace metadata dict (source_field -> list of {'value': ...} dicts)
    plus a downloads list (from get_original_bundle_downloads) to an ardvark_data
    record. Used by build_ardvark_record() during the fresh API fetch / mapping step.

    Note: locn_geometry, dcat_bbox, and dct_references_s are intentionally left
    blank here. They depend on fields (box_* / url_download_*) that a reviewer
    may hand-edit in aardvark_data_for_review.xlsx, so they are calculated only
    after that file is re-ingested (see the re-ingest cell), not at mapping time.
    """
    # Start from a fresh copy of the empty template for this item
    record = {field: '' for field in field_names}

    # dspace_UUID is kept for filename purposes only (dropped from the final JSON).
    record['dspace_UUID'] = uuid

    # id: "umass-sw-data-" prepended to the digits after the last "/" in
    # dc.identifier.uri (e.g. "https://hdl.handle.net/20.500.14394/58970" ->
    # "umass-sw-data-58970"). This is handled explicitly rather than via
    # mapping_rules because dc.identifier.uri is also mapped to url_dataset in
    # dc_ard_map.txt, and mapping_rules only keeps one target per source field.
    # Falls back to the dspace UUID (also prefixed) if dc.identifier.uri is
    # missing, so "id" (required by the schema) is always populated.
    id_prefix = "umass-sw-data-"
    identifier_uri_values = [
        v.get('value') for v in metadata.get('dc.identifier.uri', []) if v.get('value') is not None
    ]
    if identifier_uri_values:
        numeric_id = identifier_uri_values[0].rstrip('/').rsplit('/', 1)[-1]
    else:
        numeric_id = uuid
    record['id'] = id_prefix + numeric_id

    # Apply mappings from the dc_ard_map.txt file
    for source, target in mapping_rules.items():
        if source in metadata and metadata[source]:
            values = [item.get('value') for item in metadata[source] if item.get('value') is not None]

            if target in ['dct_creator_sm', 'dct_subject_sm']:
                # Multi-valued string fields, join with '; '
                record[target] = '; '.join(values)
            elif target == 'dct_identifier_sm':
                current_identifiers = record.get(target, '').split('; ') if record.get(target) else []
                for val in values:
                    if val not in current_identifiers:
                        current_identifiers.append(val)
                record[target] = '; '.join(filter(None, current_identifiers))
            elif target == 'dct_language_sm':
                if values:
                    lang_code = values[0]
                    record[target] = lang_code.split('_')[0] if '_' in lang_code else lang_code
            elif target == 'gbl_indexYear_im':
                if values:
                    try:
                        record[target] = int(values[0].split('-')[0])
                    except (ValueError, IndexError):
                        record[target] = ''
            else:
                if values:
                    record[target] = values[0]

    # --- Add hardcoded/derived fields not covered by dc_ard_map.txt ---

    if not record.get('dct_alternative_sm') and record.get('dct_title_s'):
        record['dct_alternative_sm'] = record['dct_title_s']

    if not record.get('dct_title_s') and record.get('dct_alternative_sm'):
        record['dct_title_s'] = 'Data for:' + record['dct_alternative_sm']

    if not record.get('schema_provider_s'):
        record['schema_provider_s'] = 'UMass'

    if not record.get('gbl_mdVersion_s'):
        record['gbl_mdVersion_s'] = 'Aardvark'

    if not record.get('dct_publisher_sm'):
        record['dct_publisher_sm'] = 'University of Massachusetts Amherst'

    #if not record.get('dct_format_s'):
    #    record['dct_format_s'] = 'Raster Dataset'

    if not record.get('dct_accessRights_s'):
        record['dct_accessRights_s'] = 'Public'

    if not record.get('dct_language_sm'):
        record['dct_language_sm'] = 'eng' 

    if not record.get('pcdm_memberOf_sm'):
        record['pcdm_memberOf_sm'] = 'umass-sw-data01' 

    if not record.get('gbl_resourceClass_sm'):
        record['gbl_resourceClass_sm'] = 'Datasets'     
    

    # gbl_resourceClass_sm values use the OGM Aardvark controlled vocabulary,
    # which is plural: "Datasets", "Publications" (not an official term but
    # kept here as-is pending a real mapping), "Other", etc.
    # if not record.get('gbl_resourceClass_sm') and 'dc.type' in metadata and metadata['dc.type']:
    #     resource_type = metadata['dc.type'][0].get('value', '')
    #     if resource_type:
    #         if resource_type in ['Article', 'Conference Paper', 'Review', 'Publication']:
    #             record['gbl_resourceClass_sm'] = 'Publications'
    #         elif resource_type == 'Dataset':
    #             record['gbl_resourceClass_sm'] = 'Datasets'
    #         elif resource_type == 'Code':
    #             record['gbl_resourceClass_sm'] = 'Code'
    #         else:
    #             record['gbl_resourceClass_sm'] = 'Other'

    # locn_geometry / dcat_bbox are NOT computed here anymore -- see the
    # re-ingest cell, which recomputes them from box_* after manual editing.

    # url_download_1_link/type, url_download_2_link/type, ... one pair per
    # bitstream found in the ORIGINAL bundle (not capped at 2; a record may
    # end up with more numbered pairs than field_names defines). Also sum
    # sizeBytes across all ORIGINAL bitstreams into gbl_fileSize_s, in MB.
    total_bytes = 0
    for i, (content_url, extension, size_bytes) in enumerate(downloads, start=1):
        record[f'url_download_{i}_link'] = content_url
        record[f'url_download_{i}_type'] = extension
        total_bytes += size_bytes

    if downloads:
        total_mb = total_bytes / (1024 * 1024)
        record['gbl_fileSize_s'] = f"{total_mb:.2f} MB"

    # dct_references_s is NOT computed here anymore -- see the re-ingest cell,
    # which recomputes it from url_dataset and url_download_*_link/type after
    # manual editing.

    return record


def build_ardvark_record(item_id):
    """Fetch a single item from the repository API and map it to an ardvark_data record."""
    itemurl = repository.rstrip('/') + "/server/api/core/items/" + item_id
    response = requests.get(itemurl)
    response.raise_for_status()
    source_data_dict = response.json()
    metadata = source_data_dict.get('metadata', {})
    downloads = get_original_bundle_downloads(source_data_dict)

    return map_metadata_to_ardvark(
        uuid=source_data_dict.get('uuid', ''),
        metadata=metadata,
        downloads=downloads,
    )

## Batch processing: read item IDs from a file

Reads item IDs from a plain-text file (one UUID per line, blank lines and, fetches and maps each one using `build_ardvark_record`, and exports all resulting rows to a single Excel file.

In [ ]:
item_ids_path = "item_ids.txt"

with open(item_ids_path, "r") as f:
    item_ids = [
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith("#")
    ]

print(f"Loaded {len(item_ids)} item ID(s) from {item_ids_path}")

In [ ]:
records = []
failed_ids = []

for item_id in item_ids:
    time.sleep(1)
    try:
        records.append(build_ardvark_record(item_id))
    except requests.exceptions.RequestException as e:
        print(f"Failed to fetch item {item_id}: {e}")
        failed_ids.append(item_id)

print(f"Successfully processed {len(records)} of {len(item_ids)} item(s)")
if failed_ids:
    print(f"Failed IDs: {failed_ids}")

In [ ]:
#Export mapped records for manual review/editing (before JSON creation)
df_batch = pd.DataFrame(records)
review_output_path = "aardvark_data_for_review.xlsx"
df_batch.to_excel(review_output_path, index=False)

print(f"Exported {len(df_batch)} mapped record(s) to {review_output_path}")
print("Download and edit this file as needed, then run the re-ingest cell below.")

## Manual-edit workflow: edit the mapped data, then create JSON

The records above have already been mapped from DSpace metadata to Aardvark fields
and exported to `aardvark_data_for_review.xlsx`. Download that file, correct any
values by hand -- including derived fields like `dct_title_s`, `locn_geometry`, or
`gbl_resourceClass_sm` -- save it, then run the re-ingest cell below to load your
edits before the final JSON files are created.

Do not add or remove columns. Multi-valued fields (columns ending `_sm`/`_im`)
should stay `; `-separated, matching how they were exported.

In [ ]:
#Re-ingest the (possibly hand-edited) mapped review file
edited_review_path = "aardvark_data_for_review.xlsx"
#edited_review_path = "Copy of aardvark_data_for_review.xlsx"

df_reingested = pd.read_excel(edited_review_path)

# Recompute locn_geometry / dcat_bbox from the box_* fields, in case they were
# edited by hand. This mirrors the ENVELOPE(west, east, north, south) logic
# that used to run at mapping time. Rows without all four box_* values end up
# with an empty geometry/bbox, even if a stale value was present before editing.
bbox_fields = ['box_westlimit', 'box_eastlimit', 'box_northlimit', 'box_southlimit']

def recompute_bbox_geometry(row):
    values = [row.get(f) for f in bbox_fields]
    if all(pd.notna(v) and str(v).strip() != '' for v in values):
        return "ENVELOPE({}, {}, {}, {})".format(*[str(v).strip() for v in values])
    return ''

recomputed_geometry = df_reingested.apply(recompute_bbox_geometry, axis=1)
df_reingested['locn_geometry'] = recomputed_geometry
df_reingested['dcat_bbox'] = recomputed_geometry

# Recompute dcat_centroid, in case the source fields were edited by hand. Per
# the OGM Aardvark spec, dcat_centroid is "latitude,longitude". Prefer
# point_north/point_east when both are present (these already are a single
# lat/lon point), falling back to the midpoint of the box_* fields
# (north/south for latitude, west/east for longitude) otherwise.
def recompute_centroid(row):
    point_north = row.get('point_north')
    point_east = row.get('point_east')
    if (
        pd.notna(point_north) and str(point_north).strip() != ''
        and pd.notna(point_east) and str(point_east).strip() != ''
    ):
        try:
            latitude = float(str(point_north).strip())
            longitude = float(str(point_east).strip())
            return f"{latitude:.6f},{longitude:.6f}"
        except ValueError:
            pass

    bbox_values = [row.get(f) for f in bbox_fields]
    if all(pd.notna(v) and str(v).strip() != '' for v in bbox_values):
        try:
            west, east, north, south = (float(str(v).strip()) for v in bbox_values)
        except ValueError:
            return ''
        latitude = (north + south) / 2
        longitude = (west + east) / 2
        return f"{latitude:.4f},{longitude:.4f}"

    return ''

recomputed_centroid = df_reingested.apply(recompute_centroid, axis=1)
df_reingested['dcat_centroid'] = recomputed_centroid

# Recompute dct_references_s from url_dataset and url_download_*_link/_type,
# in case those were edited by hand (or bitstreams added/removed). This
# mirrors the dct_references_s logic that used to run at mapping time.
download_link_cols = sorted(
    (col for col in df_reingested.columns if re.match(r'^url_download_\d+_link$', col)),
    key=lambda c: int(re.search(r'\d+', c).group()),
)

def recompute_references(row):
    url_dataset = row.get('url_dataset', '')
    url_dataset = '' if pd.isna(url_dataset) else str(url_dataset).strip()

    download_entries = []
    for link_col in download_link_cols:
        idx = re.search(r'\d+', link_col).group()
        link_val = row.get(link_col, '')
        if pd.isna(link_val) or str(link_val).strip() == '':
            continue
        type_val = row.get(f'url_download_{idx}_type', '')
        type_val = '' if pd.isna(type_val) else str(type_val).strip()
        download_entries.append({"url": str(link_val).strip(), "label": type_val})

    references = {
        "http://schema.org/url": url_dataset,
        "http://schema.org/downloadUrl": download_entries,
    }
    return json.dumps(references, separators=(',', ':'))

df_reingested['dct_references_s'] = df_reingested.apply(recompute_references, axis=1)

print(f"Loaded {len(df_reingested)} record(s) from {edited_review_path} for JSON export")
print(f"Recomputed locn_geometry/dcat_bbox for {(recomputed_geometry != '').sum()} of {len(df_reingested)} record(s) from box_* fields")
print(f"Recomputed dcat_centroid for {(recomputed_centroid != '').sum()} of {len(df_reingested)} record(s) (point_north/point_east preferred, box_* fallback)")
print("Recomputed dct_references_s for all records from url_dataset/url_download_*_link/_type")

In [ ]:
#Convert the (possibly hand-edited) aardvark_data_for_review.xlsx into one Aardvark
#JSON file per record, named <id>.json (the Aardvark "id" field, not the dspace
#UUID), in the aardvark_json/ folder.
import os
from datetime import datetime, timezone

def row_to_aardvark_json(record):
    """Convert one ardvark_data record (dict of strings/NaN) into a properly
    typed Aardvark JSON dict: multi-valued fields (suffix _sm/_im) become
    lists, booleans (_b) become bool, and empty values are omitted entirely
    (per Aardvark convention). Fields starting with "box" or "url", any
    reviewer-added column starting with "change" (case-insensitive, e.g.
    "change_notes"), the "geospatial" and "DSL" columns, and the dspace_UUID
    field are all dropped from the output entirely (dspace_UUID is only used
    internally, not included in the final JSON), and id is lowercased.
    """
    aardvark_json = {}
    excluded_exact = {'geospatial', 'dsl', 'point_north', 'point_east' }

    for field, value in record.items():
        if pd.isna(value) or value == '':
            continue
        field_lower = field.lower()
        if field == 'dspace_UUID' or field.startswith(('box', 'url')):
            continue
        if field_lower.startswith('change') or field_lower in excluded_exact:
            continue

        if field == 'id':
            aardvark_json[field] = str(value).lower()
        elif field.endswith(('_sm', '_im')):
            parts = [p.strip() for p in str(value).split(';') if p.strip()]
            if field.endswith('_im'):
                parts = [int(p) for p in parts]
            aardvark_json[field] = parts
        elif field.endswith('_b'):
            aardvark_json[field] = str(value).strip().lower() in ('true', '1', 'yes')
        elif field == 'gbl_indexYear_im':
            aardvark_json[field] = [int(value)]
        else:
            aardvark_json[field] = value

    return aardvark_json


json_output_dir = "aardvark_json"
os.makedirs(json_output_dir, exist_ok=True)

json_paths = []
for record in df_reingested.to_dict(orient="records"):
    record_id = str(record.get('id', '')).strip()
    if not record_id or record_id.lower() == 'nan':
        print(f"Skipping record with missing id: {record.get('dspace_UUID', '')}")
        continue

    # Sanitize in case "id" ever contains characters unsafe for filenames
    # (e.g. "/"), even though the current mapping shouldn't produce any.
    safe_filename = re.sub(r'[\\/:*?"<>|]', '-', record_id.lower())

    aardvark_json = row_to_aardvark_json(record)

    # gbl_mdModified_dt: stamp with the current UTC time at JSON creation, in
    # the W3C Date and Time Format required by the schema (YYYY-MM-DDThh:mm:ssZ).
    aardvark_json['gbl_mdModified_dt'] = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

    json_path = os.path.join(json_output_dir, f"{safe_filename}.json")
    with open(json_path, 'w') as f:
        json.dump(aardvark_json, f, indent=2)
    json_paths.append(json_path)

print(f"Wrote {len(json_paths)} Aardvark JSON file(s) to {json_output_dir}/")

In [ ]:
#Validate the generated Aardvark JSON files against the official OGM Aardvark JSON Schema
# import glob

# import jsonschema

# schema_url = "https://opengeometadata.org/schema/geoblacklight-schema-aardvark.json"
# aardvark_schema = requests.get(schema_url).json()

# validator = jsonschema.Draft7Validator(aardvark_schema)

# for path in sorted(glob.glob(os.path.join(json_output_dir, "*.json"))):
#     with open(path) as f:
#         doc = json.load(f)

#     errors = sorted(validator.iter_errors(doc), key=lambda e: e.path)
#     print(f"\n{path}")
#     if not errors:
#         print("  OK")
#     else:
#         for error in errors:
#             location = "/".join(str(p) for p in error.path) or "(root)"
#             print(f"  [{location}] {error.message}")